# Animated noodle plot

Loads a CSV file with a data. frame of collective events created by `arcos4py.track_collective_events` and saves individual frames of the noodle plot. The frames can then be used to create an animated plot.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from multiprocessing import Pool
from tqdm import tqdm
import joblib

In [ ]:
# Colour palette
TAB20 = [
    "#1f77b4",
    "#aec7e8",
    "#ff7f0e",
    "#ffbb78",
    "#2ca02c",
    "#98df8a",
    "#d62728",
    "#ff9896",
    "#9467bd",
    "#c5b0d5",
    "#8c564b",
    "#c49c94",
    "#e377c2",
    "#f7b6d2",
#    "#7f7f7f",
#    "#c7c7c7",
    "#bcbd22",
    "#dbdb8d",
    "#17becf",
    "#9edae5",
]

from datetime import timedelta
from string import Template
from typing import List, Tuple, Union

from functools import partial

PXSZ = 332.80/1024

def do_plot(df_filtered, NoodlePlot, update_hlines, i):
    # set black plot theme
    plt.style.use("dark_background")
    plt.rcParams["font.family"] = "Arial"
    plt.rcParams["figure.dpi"] = 300
    plt.rcParams["font.size"] = 16
    plt.rcParams["axes.labelsize"] = 12
    plt.rcParams["axes.titlesize"] = 12
    plt.rcParams["legend.fontsize"] = 12
    plt.rcParams["xtick.labelsize"] = 12
    plt.rcParams["ytick.labelsize"] = 12

    plt.rcParams["axes.linewidth"] = 2
    plt.rcParams["xtick.major.width"] = 2
    plt.rcParams["ytick.major.width"] = 2


    px = 1 / plt.rcParams["figure.dpi"]  # pixel in inches
    fig, ax = plt.subplots(figsize=(1024 * px, 1024 * px))

    line_col = plt.vlines(i, 0, df_filtered.y.max() + 10, color="white", linestyle="-", lw=1, alpha=0.8)

    _, ax = NoodlePlot(df_filtered[df_filtered.t <= i], "cl_id", "track_id", "t", "x", "y").plot(
        fig, ax, "y"
    )
    ax.set_ylim(0, df_filtered.y.max() + 10)
    ax.set_xlim(0, 300)
    update_hlines(line_col, x=[i], ymin=0, ymax=df_filtered.y.max() + 10)

    # x_labels_new = [strfdelta(timedelta(seconds=int(s) * frametime), "%D:%H:%M") for s in x_labels]
    # plt.xticks(x_labels, x_labels_new, size=12)
    plt.yticks(size=12)
    # ax.xaxis.set_label_position('top')
    # ax.xaxis.tick_top()
    labelsX = np.arange(0, 360, step=60) * 5 / 60
    labelsX = labelsX.astype(int)

    ax.set_xticks(np.arange(0, 360, step=60))
    ax.set_xticklabels(labelsX, family="Arial")

    # reverse y-axis
    ax.set_ylim(0, df_filtered.y.max() + 10)
    ax.invert_yaxis()

    plt.tight_layout(pad=0.2)
    plt.savefig(f"noodles/{i}.tif", dpi=300)
    plt.close()


class NoodlePlot:
    """Create Noodle Plot of cell tracks, colored by collective event id.

    Attributes:
        df (pd.DataFrame): DataFrame containing collective events from arcos.
        colev (str): Name of the collective event column in df.
        trackid (str): Name of the track column in df.
        frame (str): Name of the frame column in df.
        posx (str): Name of the X coordinate column in df.
        posy (str): Name of the Y coordinate column in df.
        posz (str): Name of the Z coordinate column in df,
            or None if no z column.
    """

    def __init__(
        self,
        df: pd.DataFrame,
        colev: str,
        trackid: str,
        frame: str,
        posx: str,
        posy: str,
        posz: Union[str, None] = None,
    ):
        """Constructs class with given parameters.

        Arguments:
            df (pd.DataFrame): DataFrame containing collective events from arcos.
            colev (str): Name of the collective event column in df.
            trackid (str): Name of the track column in df.
            frame (str): Name of the frame column in df.
            posx (str): Name of the X coordinate column in df.
            posy (str): Name of the Y coordinate column in df.
            posz (str | None): Name of the Z coordinate column in df,
                or None if no z column.
        """
        self.df = df
        self.colev = colev
        self.trackid = trackid
        self.frame = frame
        self.posx = posx
        self.posy = posy
        self.posz = posz

    def _prepare_data_noodleplot(
        self,
        df: pd.DataFrame,
        color_cylce: list[str],
        colev: str,
        trackid: str,
        frame: str,
        posx: str,
        posy: str,
        posz: Union[str, None] = None,
    ) -> tuple[list[np.ndarray], np.ndarray]:
        """From arcos collective event data,\
        generates a list of numpy arrays, one for each event.

        Arguments:
            df (pd.DataFrame): DataFrame containing collective events from arcos.
            color_cylce (list[str]): list of colors used to color trackid's
                for individual collective events.
            colev (str): Name of the collective event column in df.
            trackid (str): Name of the track column in df.
            frame: (str): Name of the frame column in df.
            posx (str): Name of the X coordinate column in df.
            posy (str): Name of the Y coordinate column in df.
            posz (str): Name of the Z coordinate column in df,
                or None if no z column.

        Returns:
            grouped_array (list[np.ndarray]): List of collective events data
            colors (np.ndarray): colors for each collective event.
        """
        df = df.copy()
        # factorize trackid to get unique values and make sure they are nummeric
        df[trackid] = df[trackid].factorize()[0]
        # sort by collective event and trackid
        df = df.sort_values([colev, trackid])
        if posz:
            array = df[[colev, trackid, frame, posx, posy, posz]].to_numpy()
        else:
            array = df[[colev, trackid, frame, posx, posy]].to_numpy()
        # generate goroups for each unique value
        grouped_array = np.split(array, np.unique(array[:, 0], axis=0, return_index=True)[1][1:])
        # make collids sequential
        seq_colids = np.concatenate(
            [np.repeat(i, value.shape[0]) for i, value in enumerate(grouped_array)],
            axis=0,
        )
        array_seq_colids = np.column_stack((array, seq_colids))
        # split sequential collids array by trackid and collid
        grouped_array = np.split(
            array_seq_colids,
            np.unique(array_seq_colids[:, :2], axis=0, return_index=True)[1][1:],
        )
        # generate colors for each collective event, wrap arround the color cycle
        colors = np.take(np.array(color_cylce), [i + 1 for i in np.unique(seq_colids)], mode="wrap")
        return grouped_array, colors

    def _create_noodle_plot(self, ax, grouped_data: list[np.ndarray], colors: np.ndarray):
        """Plots the noodle plot."""
        # fig, ax = plt.subplots()
        ax.set_xlabel("Time [hours]")
        ax.set_ylabel("Position Y [µm]")
        for dat in grouped_data:
            if dat.size == 0:
                continue
            ax.plot(
                dat[:, 2],
                dat[:, self.projection_index],
                c=colors[int(dat[0, -1])],
            )
        return ax

    def plot(self, fig, ax, projection_axis: str, color_cylce: List[str] = TAB20, frametime: float = 1):
        """Create Noodle Plot of cell tracks, colored by collective event id.

        Arguments:
            projection_axis (str): Specify with witch coordinate the noodle
                plot should be drawn. Has to be one of the posx, posy or posz arguments
                passed in during the class instantiation process.
            color_cylce (list[str]): List of hex color values or string names
                (i.e. ['red', 'yellow']) used to color collecitve events. Cycles through list.

        Returns:
            fig (matplotlib.figure.Figure): Matplotlib figure object for the noodle plot.
            axes (matplotlib.axes.Axes): Matplotlib axes for the nooble plot.
        """
        if projection_axis not in [self.posx, self.posy, self.posz]:
            raise ValueError(f"projection_axis has to be one of {[self.posx, self.posy, self.posz]}")
        if projection_axis == self.posx:
            self.projection_index = 3
        elif projection_axis == self.posy:
            self.projection_index = 4
        elif projection_axis == self.posz:
            self.projection_index = 5
        grpd_data, colors = self._prepare_data_noodleplot(
            self.df, color_cylce, self.colev, self.trackid, self.frame, self.posx, self.posy, self.posz
        )
        axes = self._create_noodle_plot(ax, grpd_data, colors)
        return fig, axes


def update_vlines(h, x, ymin=None, ymax=None):
    seg_old = h.get_segments()
    if ymin is None:
        ymin = seg_old[0][0, 1]
    if ymax is None:
        ymax = seg_old[0][1, 1]

    seg_new = [np.array([[xx, ymin], [xx, ymax]]) for xx in x]

    h.set_segments(seg_new)


def update_hlines(h, x, ymin=None, ymax=None):
    seg_old = h.get_segments()
    if ymin is None:
        ymin = seg_old[0][0, 0]
    if ymax is None:
        ymax = seg_old[0][1, 0]

    seg_new = [np.array([[xx,ymin], [xx, ymax]]) for xx in x]

    h.set_segments(seg_new)

In [ ]:
df = pd.read_csv("output-data/dfCE_filtered.csv.gz")
df_filtered = df
df_filtered[['x','y']] = df_filtered[['x','y']] * PXSZ
print('start plotting')
with joblib.Parallel(n_jobs=-1) as parallel:
    parallel(joblib.delayed(partial(do_plot, df_filtered, NoodlePlot, update_hlines))(i) for i in tqdm(range(0, 300), total=300))